**Connect Scripts**

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from Scripts import FileHandler as fh

**Load Data Set**

In [ ]:
# load data set
import kagglehub
from pathlib import Path

downloadPath = kagglehub.dataset_download('lespin/house-prices-dataset')
dataPath = Path(downloadPath)

print(f'Content of {dataPath}:')
for item in dataPath.iterdir():
    print(f"    -{item.name} ({'Folder' if item.is_dir() else 'File'})")

**Inspect Data Set**

In [ ]:
# Inspect data set
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

dfHousing = pd.read_csv(f'{downloadPath}/train.csv')

# print first 5 lines
display(dfHousing.head())

# check data set shape
print(f"\n Housing Dataset shape : {dfHousing.shape}")

# check available data types
print("\n Data Type Count")
print(dfHousing.dtypes.value_counts())

# check missing values in data set
print("\n Missing values in Data set")
dfHousing.info()

# statistical summary
print("\n statistical summary")
display(dfHousing['SalePrice'].describe())

# check distribution
sns.set_theme(style='whitegrid')
plt.figure(figsize=(8,5))
sns.histplot(dfHousing['SalePrice'], kde=True, color='green', bins=30)
plt.title('Distribution of House sale prices')
plt.xlabel('Sale Price/($)')
plt.ylabel('Count')
plt.show()


**Train/Validate/Test Split**
- avoid cross contamination (don't need to get influence by Training data)
- 70% : Train ; 15% : Validation ; 15% : Test

In [ ]:
from sklearn.model_selection import train_test_split

dfTrain, dfTemp = train_test_split(dfHousing, test_size=0.30, random_state=42)
dfVal, dfTest = train_test_split(dfTemp, test_size=0.5, random_state=42)

print(f"Dataset Size: {len(dfHousing)} | Train Size: {len(dfTrain)} | Validate Size: {len(dfVal)} | Test Size: {len(dfTest)}")
fh.saveDataSets(dfTrain, dfVal, dfTest, "housingdataset/preprocessing", "01_datasplit")

**Handle Missing Values**
- Drop the columns
- Imputation/ Replace
    - mean
    - meadian
    - frequent
    - null value

In [ ]:
dfTrain, dfVal, dfTest = fh.loadDataSets("housingdataset/preprocessing", "01_datasplit")

In [ ]:
# check missing values
missingTrain = dfTrain.isnull().sum()
missingVal = dfVal.isnull().sum()
missingTest = dfTest.isnull().sum()

print("Train set")
print(missingTrain[missingTrain>0])

print("\nVal set")
print(missingVal[missingVal>0])

print("\nTest set")
print(missingTest[missingTest>0])

In [ ]:
# drop columns
dropList = []

print(f"before - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")
dfTrain = dfTrain.drop(columns=dropList, errors='ignore')
dfVal = dfVal.drop(columns=dropList, errors='ignore')
dfTest = dfVal.drop(columns=dropList, errors='ignore')
print(f"after - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")

In [ ]:
# replace with median value
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
medianReplaceList = ['LotFrontage']

for column in medianReplaceList:
    if column in dfTrain.columns:
        imputer.fit(dfTrain[[column]])
        print(f"Imputed(median) value: {imputer.statistics_}")

        dfTrain[[column]] = imputer.transform(dfTrain[[column]])

        if column in dfVal:
            dfVal[[column]] = imputer.transform(dfVal[[column]])

        if column in dfTest:
            dfTest[[column]] = imputer.transform(dfTest[[column]])

print("missing in Train: ", dfTrain['LotFrontage'].isnull().sum())

In [ ]:
# replace with frequent value

frequentReplaceList = ['Electrical']

for column in frequentReplaceList:
    if column in dfTrain.columns:
        modeValue = dfTrain[column].mode()[0]
        print(f"Imputed(frequent) value: {modeValue}")

        dfTrain[column] = dfTrain[column].fillna(modeValue)

        if column in dfVal:
            dfVal[column] = dfVal[column].fillna(modeValue)

        if column in dfTest:
            dfTest[column] = dfTest[column].fillna(modeValue)


print("missing in Train: ", dfTrain['Electrical'].isnull().sum())

In [ ]:
catCols = dfTrain.select_dtypes(include=['str']).columns
numCols = dfTrain.select_dtypes(exclude=['str']).columns

catNullList = dfTrain[catCols].columns[dfTrain[catCols].isnull().any()].tolist()
numNullList = dfTrain[numCols].columns[dfTrain[numCols].isnull().any()].tolist()

print(catNullList)
print(numNullList)


In [ ]:
# replace with none/0 value

# fill numerical columns with 0
dfTrain[numNullList] = dfTrain[numNullList].fillna(0)
dfVal[numNullList] = dfVal[numNullList].fillna(0)
dfTest[numNullList] = dfTest[numNullList].fillna(0)

# fill categorical columns with 'None'
dfTrain[catNullList] = dfTrain[catNullList].fillna("None")
dfVal[catNullList] = dfVal[catNullList].fillna("None")
dfTest[catNullList] = dfTest[catNullList].fillna("None")

In [ ]:
# check missing values
missingTrain = dfTrain.isnull().sum()
missingVal = dfVal.isnull().sum()
missingTest = dfTest.isnull().sum()

print("Train set")
print(missingTrain[missingTrain>0])

print("\nVal set")
print(missingVal[missingVal>0])

print("\nTest set")
print(missingTest[missingTest>0])

In [ ]:
fh.saveDataSets(dfTrain, dfVal, dfTest, "housingdataset/preprocessing", "02_handlemissingvalues")

**Detect Outliers**
- Not Need to handle:
    - predefined ranges
    - Years
    - Months
    - Timestamps
- Required to handle:
    - calculated
    - physical
- May be:
    - counts
    - Targets  
      
        
- Outlier finding methods:
    - Q1:first quartile - 25% of data falls below this value
    - Q2:second quartile - split data in half
    - Q3:third quartile - 75% of data falls below this value
    - IQR = Q3 -Q1:interquartile value: middle 50% of the spread
    - Oulier boundary: 
        - lower fence: Q1 - 1.5*IQR
        - upper fence: Q3 + 1.5*IQR

In [ ]:
dfTrain, dfVal, dfTest = fh.loadDataSets("housingdataset/preprocessing", "02_handlemissingvalues")

In [ ]:
numCol = dfTrain.select_dtypes(exclude=["str"]).columns
print(numCol)

In [ ]:
outlierCheckList = [
        'LotFrontage', 'LotArea', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 
        'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 
        'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
        'Fireplaces', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', 
        '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'SalePrice']

validColmns = [col for col in outlierCheckList if col in dfTrain.columns]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

nCols = 3
nRows = int(np.ceil(len(validColmns)/nCols))

fig, axes = plt.subplots(nRows, nCols, figsize=(15, 3.5 * nRows))
axes = axes.flatten()

for idx, col in enumerate(validColmns):
    ax = axes[idx]
    ax.plot(dfTrain.index, dfTrain[col], marker='o', linestyle='', alpha=0.5, markersize=4)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.tick_params(labelsize=8)
    ax.grid(True, linestyle='--', alpha=0.3)

for idx in range(len(validColmns), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.suptitle('Outlier check: scatterplots by column', fontsize=14, y=1.01)
plt.show()

In [ ]:
for col in outlierCheckList:
    if col not in dfTrain.columns:
        continue

    Q1 = dfTrain[col].quantile(0.25)
    Q3 = dfTrain[col].quantile(0.75)
    IQR = Q3-Q1

    lowerFence = Q1 - 1.5 * IQR
    upperFence = Q3 + 1.5 * IQR

    outlierRows = dfTrain[(dfTrain[col] < lowerFence) | (dfTrain[col] > upperFence)]
    outlierValues = outlierRows[col].sort_values()

    print(f"\ncolumn: {col}")
    print(f"Q1={Q1:.2f} | Q3={Q3:.2f} | IQR={IQR:.2f}")
    print(f"Fences: [{lowerFence:.2f}, {upperFence:.2f}]")
    print(f"Outlier Count: {len(outlierValues)} | Percentage: {(len(outlierValues)/ len(dfTrain)) * 100}")

    if len(outlierValues) > 0:
        lowOutliers = outlierValues[outlierValues < lowerFence]
        highOutliers = outlierValues[outlierValues > upperFence]

        if len(lowOutliers) > 0:
            print(f"    below lower fence: {list(lowOutliers)}")

        if len(highOutliers) > 0:
            print(f"    above upper fence: {list(highOutliers)}")
    else:
        print("no outliers found")

In [ ]:
# set manual bounds
# use domain knowledge

investigateUpper = {
    'LotFrontage' :  300,
    'LotArea' :  100000,
    'MasVnrArea' : 1000,
    'BsmtFinSF1': 5000,
    'TotalBsmtSF': 6000,
    '1stFlrSF' : 4000,
    'GrLivArea': 4000,
    'WoodDeckSF': 600,
    'OpenPorchSF': 400,
    'EnclosedPorch': 500,
    '3SsnPorch': 400,
    'SalePrice': 600000
}

In [ ]:
for col, val in investigateUpper.items():
    rows = dfTrain[dfTrain[col] > val]
    print(f"{col} Value: {val} : {len(rows)} rows")
    print(rows[['OverallQual', 'OverallCond', 'GrLivArea', 'LotArea', 'SaleCondition', 'SaleType', 'SalePrice']].to_string())

In [ ]:
# define which columns need capping
capList = ['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch']

caps = {}

for colmn in capList:
    if colmn in dfTrain.columns:
        caps[colmn] = dfTrain[colmn].quantile(0.995) # use 99.5% from train set
        print(f"{colmn}: cap = {caps[colmn]}")

In [ ]:
def applyOutlierTreatment(df, caps, dataset_name:str):
    df = df.copy()

    for colmn, cap in caps.items():
        if colmn in df.columns:
            before = (df[colmn] > cap).sum()
            df[colmn] = df[col].clip(upper=cap)
            if before > 0:
                print(f"{dataset_name}: capped {before} values in {colmn} at {cap}")

    return df

dfTrain = applyOutlierTreatment(dfTrain, caps, "Train")

In [ ]:
rows = dfTrain[(dfTrain['WoodDeckSF'] > 630) | (dfTrain['EnclosedPorch'] > 291)]
print(rows.to_string())

In [ ]:
fh.saveDataSets(dfTrain, dfVal, dfTest, "housingdataset/preprocessing", "03_handleoutliers")

**Feature Engineering**
- introduce usefull features
    - create
    - transform
    - combine

In [ ]:
dfTrain, dfVal, dfTest = fh.loadDataSets("housingdataset/preprocessing", "03_handleoutliers")

In [ ]:
addColList = dfTrain.columns[(dfTrain == "None").any() | (dfTrain == 0).any()].tolist()
print(addColList)

In [ ]:
# create: add new feature which has very valuable binary data
addCreateColumns = {
    'HasAlley' : 'Alley',
    'HasVnr' : 'MasVnrType',
    'HasBsmt' : 'BsmtQual',
    'HasKitchenAbvGr' : 'KitchenAbvGr',
    'HasFireplaces' : 'FireplaceQu',
    'HasGarage' : 'GarageType',
    'HasPool' : 'PoolQC',
    'HasFence' : 'Fence',
    'HasMisc' : 'MiscFeature'
}

print(f"before - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")
for key, val in addCreateColumns.items():
    if key not in dfTrain.columns:
        dfTrain[key] = (dfTrain[val] != 'None').astype(int)

    if key not in dfVal.columns:
        dfVal[key] = (dfVal[val] != 'None').astype(int)

    if key not in dfTest.columns:
        dfTest[key] = (dfTest[val] != 'None').astype(int)
print(f"after - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")

In [ ]:
# tranform: age has more impact than raw year representaiton
transformColumns = {
    'HouseAge' : ['YrSold', 'YearBuilt'],
    'EffectiveAge' : ['YrSold', 'YearRemodAdd'],
    'GarageAge' : ['YrSold', 'GarageYrBlt']
}

print(f"before - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")
for key, lst in transformColumns.items():
    yrSold = lst[0]
    yrAdd = lst[1]

    if key not in dfTrain.columns:
        dfTrain[key] = np.where(dfTrain[yrAdd]>0, dfTrain[yrSold] - dfTrain[yrAdd], 0)

    if key not in dfVal.columns:
        dfVal[key] = np.where(dfVal[yrAdd]>0, dfVal[yrSold] - dfVal[yrAdd], 0)

    if key not in dfTest.columns:
        dfTest[key] = np.where(dfTest[yrAdd]>0, dfTest[yrSold] - dfTest[yrAdd], 0)
print(f"after - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")

In [ ]:
# combine: total area
combineColumns = {
    'TotalPorchSF' : ['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'WoodDeckSF'],
    'TotalHouseSF' : ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF']
}

print(f"before - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")
for key, lst in combineColumns.items():
    if key not in dfTrain.columns:
        dfTrain[key] = dfTrain[lst].fillna(0).sum(axis=1)

    if key not in dfVal.columns:
        dfVal[key] = dfVal[lst].fillna(0).sum(axis=1)

    if key not in dfTest.columns:
        dfTest[key] = dfTest[lst].fillna(0).sum(axis=1)
print(f"after - train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")

In [ ]:
display(dfTrain[['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'WoodDeckSF', 'TotalPorchSF']].head())
display(dfTrain[['TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'TotalHouseSF']].head())

In [ ]:
fh.saveDataSets(dfTrain, dfVal, dfTest, "housingdataset/preprocessing", "04_featureengineering")

**Encode Categorical Variable**
- Computers can only understand numerical values
- Encoding Methods
    - Binary Encoding
    - Ordinal Encoding
    - Nominal Encoding (onehot)

In [ ]:
dfTrain, dfVal, dfTest = fh.loadDataSets("housingdataset/preprocessing", "04_featureengineering")

In [ ]:
# find categorical columns
catCols = dfTest.select_dtypes(include=["str"]).columns

for colmn in catCols:
    uniqueValues = dfTrain[colmn].dropna().unique()
    allValues = ", ".join(map(str, uniqueValues))
    print(f"{colmn} : {allValues}")

In [ ]:
binaryList = ['CentralAir']
ordinalList = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'HeatingQC',
               'KitchenQual', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence']
nominalList = ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
               'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
               'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation',
               'BsmtFinType1', 'BsmtFinType2', 'Heating', 'Electrical', 'Functional', 'GarageType',
               'PavedDrive', 'MiscFeature', 'SaleType', 'SaleCondition']

In [ ]:
# binary --> binary encoding
for colmn in binaryList:
    print(dfTrain[colmn].value_counts())
    if colmn in dfTrain.columns:
        dfTrain[colmn] = dfTrain[colmn].map({'Y':1, 'N':0})

    if colmn in dfVal.columns:
        dfVal[colmn] = dfVal[colmn].map({'Y':1, 'N':0})

    if colmn in dfTest.columns:
        dfTest[colmn] = dfTest[colmn].map({'Y':1, 'N':0})
    print(dfTrain[colmn].value_counts())

In [ ]:
# order --> ordinal encoding
from sklearn.preprocessing import OrdinalEncoder
ordinalEncList = [['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC'],
                  ['BsmtExposure'],
                  ['GarageFinish'],
                  ['Fence']]

orderList = [['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
             ['None', 'No', 'Mn', 'Av', 'Gd'],
             ['None', 'Unf', 'RFn', 'Fin'],
             ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']]

display(dfTrain[ordinalList].head())

for i in range(len(ordinalEncList)):
    encoder = OrdinalEncoder(categories=[orderList[i]], handle_unknown='use_encoded_value', unknown_value=-1)

    for colmn in ordinalEncList[i]:
        if colmn in dfTrain.columns:
            encoder.fit(dfTrain[[colmn]])

            dfTrain[[colmn]] = encoder.transform(dfTrain[[colmn]])

        if colmn in dfVal.columns:
            dfVal[[colmn]] = encoder.transform(dfVal[[colmn]])

        if colmn in dfTest.columns:
            dfTest[[colmn]] = encoder.transform(dfTest[[colmn]])

display(dfTrain[ordinalList].head())



In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd
# nominal --> onehot encoding
print(dfTrain.shape)

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(dfTrain[nominalList])

featureNames = encoder.get_feature_names_out(nominalList)

trainOneHot = pd.DataFrame(encoder.transform(dfTrain[nominalList]), columns=featureNames, index=dfTrain.index)
valOneHot = pd.DataFrame(encoder.transform(dfVal[nominalList]), columns=featureNames, index=dfVal.index)
testOneHot = pd.DataFrame(encoder.transform(dfTest[nominalList]), columns=featureNames, index=dfTest.index)

dfTrain = pd.concat([dfTrain.drop(columns=nominalList), trainOneHot], axis=1)
dfVal = pd.concat([dfVal.drop(columns=nominalList), valOneHot], axis=1)
dfTest = pd.concat([dfTest.drop(columns=nominalList), testOneHot], axis=1)

print(dfTrain.shape)

In [ ]:
display(dfTrain.info())

In [ ]:
fh.saveDataSets(dfTrain, dfVal, dfTest, "housingdataset/preprocessing", "05_categoricalencoding")

**Scaling & Normalization**
- standarization & normalization only apply for numerical feature
    - only doing for range values, not doing for unique values
    - doing : Logistic Regression, Linear Regression, SVM, KNN, Nural Networks
    - Not   : Decision Tree, Random Forest, Gradient Boost Tree, XGBoost, LightBGM

- standarize
    $$x_{new} = \frac{x - \mu}{\sigma} $$
    - output --> no fixed range
    - Models : Linear Models, SVM

- Normalization
    $$x_{new} = \frac{x - x_{min} } {x_{max} - x_{min}}$$
    - output --> 0 - 1
    - Model : NN, Bounded input algorithms

In [ ]:
dfTrain, dfVal, dfTest = fh.loadDataSets("housingdataset/preprocessing", "05_categoricalencoding")

In [ ]:
# identify which attributes need to standarize
ignoreList = ['Id', 'SalePrice']
scaleList = []

for colmn in dfTrain.columns:
    if colmn in ignoreList:
        continue
    uniqueValues = dfTrain[colmn].dropna().unique()
    if(len(uniqueValues) > 2):
        scaleList.append(colmn)

print(scaleList)

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
display(dfTrain[scaleList].head())

normalizer = MinMaxScaler()
normalizer.fit(dfTrain[scaleList])
transformList = pd.DataFrame(normalizer.transform(dfTrain[scaleList]),columns=scaleList)

display(transformList.head())

scaler = StandardScaler()
scaler.fit(dfTrain[scaleList])

dfTrain[scaleList] = scaler.transform(dfTrain[scaleList])
dfVal[scaleList] = scaler.transform(dfVal[scaleList])
dfTest[scaleList] = scaler.transform(dfTest[scaleList])
display(dfTrain[scaleList].head())

In [ ]:
display(dfTrain[scaleList].head())
display(dfVal[scaleList].head())
display(dfTest[scaleList].head())

In [ ]:
fh.saveDataSets(dfTrain, dfVal, dfTest, "housingdataset/preprocessing","06_standardization")

**Feature Selection - Heuristic/Stateless Selection(Filter Methods)**
- choosing which features can keep or drop
- use methods relies on mathematical rule, descriptive statistics or human logic

In [ ]:
dfTrain, dfVal, dfTest = fh.loadDataSets("housingdataset/preprocessing","06_standardization")

step 1 : Drop column manually (select only useful features)

In [ ]:
def getProtectedFeatures():
    protected = [c for c in dfTrain.columns if c.startswith(('Has', 'Is', 'Was'))]
    protected.append('SalePrice')
    return protected

In [ ]:
dropList = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'MoSold', 'Id']

print(f"before - train:{dfTrain.shape} val:{dfVal.shape} test:{dfTest.shape}")
dfTrain = dfTrain.drop(columns=dropList, errors='ignore')
dfVal = dfVal.drop(columns=dropList, errors='ignore')
dfTest = dfTest.drop(columns=dropList, errors='ignore')
print(f"after - train:{dfTrain.shape} val:{dfVal.shape} test:{dfTest.shape}")

step 2 : Drop 1 dummy per ONEHOT group (dummy trap)
- why :
    - keep all ONEHOT columns meand sum along ONEHOT groups always 1 --> linear regression explodes

- fix :
    - for each ONEHOT group drop one dummy column (usually first one or most frequent one)

In [ ]:
nominalList = ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
               'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
               'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation',
               'BsmtFinType1', 'BsmtFinType2', 'Heating', 'Electrical', 'Functional', 'GarageType',
               'PavedDrive', 'MiscFeature', 'SaleType', 'SaleCondition']

oneHotGroup = {}

for oneHot in nominalList:
    attr = oneHot + '_'
    oneHotColmns = [c for c in dfTrain.columns if c.startswith(attr)]
    oneHotGroup[oneHot] = oneHotColmns

In [ ]:
print(len(nominalList))

In [ ]:
dropList = []
for group, cols in oneHotGroup.items():
    if(len(cols) > 1):
        dropList.append(cols[0])

In [ ]:
print(f"before - train:{dfTrain.shape} val:{dfVal.shape} test:{dfTest.shape}")
dfTrain = dfTrain.drop(columns=dropList, errors='ignore')
dfVal = dfVal.drop(columns=dropList, errors='ignore')
dfTest = dfTest.drop(columns=dropList, errors='ignore')
print(f"after - train:{dfTrain.shape} val:{dfVal.shape} test:{dfTest.shape}")

- ONEHOT encoded vectors sometimes carry meaningfull information
- it is unwise to remove columns which have low varience/ correlation / support / impact.
- try merge first to find out dependency.
    - ex : f_a, f_b, f_c --> f_other

In [ ]:
def updateOneHotGrps(oneHotGrps, dropped_features, dfTrain, dfVal, dfTest, verbose=False):
    rareDummyGrps = {}
    for group, cols in oneHotGrps.items():
        filtered_cols = [col for col in cols if col in dropped_features]
        if len(filtered_cols) > 0:
            rareDummyGrps[group] = filtered_cols

    if verbose:
        print("rare dummy groups")
        print(rareDummyGrps)
        print(f"\nbefore - train:{dfTrain.shape} val:{dfVal.shape} test:{dfTest.shape}")

    for group, cols in rareDummyGrps.items():
        if len(cols) > 1:
            attr = group + '_Other'
            dfTrain[attr] = dfTrain.reindex(columns=cols, fill_value=0).sum(axis=1).clip(0,1)
            dfVal[attr] = dfVal.reindex(columns=cols, fill_value=0).sum(axis=1).clip(0,1)
            dfTest[attr] = dfTest.reindex(columns=cols, fill_value=0).sum(axis=1).clip(0,1)

        dfTrain = dfTrain.drop(columns=cols, errors='ignore')
        dfVal = dfVal.drop(columns=cols, errors='ignore')
        dfTest = dfTest.drop(columns=cols, errors='ignore')

    if verbose:
        print(f"after - train:{dfTrain.shape} val:{dfVal.shape} test:{dfTest.shape}")

    return dfTrain, dfVal, dfTest

step 3 : Varience Threshold (drop near zero-varience)
- why : 
    - features that are constant (99.5% the same value) have no predictive power
    - destabilize linear model

- fix : 
    - varience
        - high varience : data points are spread out
        - low varience : data points are similar
        - zero varience : every line has exact same values

        $$\sigma^2 = \frac {\sum (x - \mu)^2}{N}$$

In [ ]:
from sklearn.feature_selection import VarianceThreshold

def findLowVarienceFeatures(dfTrain, threshold=0.005, verbose=False):
    protected = getProtectedFeatures()
    candidates = [c for c in dfTrain.columns if c not in protected]

    # remove features with low varience (<0.5%)
    selector = VarianceThreshold(threshold=threshold)
    x_candidates = dfTrain[candidates]
    x_selected = selector.fit_transform(x_candidates)

    kept_mask = selector.get_support()
    dropped_features = x_candidates.columns[~kept_mask].tolist()

    if(verbose):
        kept_features = x_candidates.columns[kept_mask].tolist()
        print(f"\nFeatures kept ({len(kept_features)}) : \n{kept_features}")
        print(f"\nFeatures dropped ({len(dropped_features)}) : \n{dropped_features}\n")

    return dropped_features

In [ ]:
itr = 0
while True:
    itr += 1
    print(f"\nIteration: {itr}")
    currentFeatures = dfTrain.shape[1]

    dropList = findLowVarienceFeatures(dfTrain, 0.005, True)
    dfTrain, dfVal, dfTest = updateOneHotGrps(oneHotGroup, dropList, dfTrain, dfVal, dfTest, True)

    newFeatures = dfTrain.shape[1]

    if currentFeatures == newFeatures:
        break

In [ ]:
print(f"before - train:{dfTrain.shape} val:{dfVal.shape} test:{dfTest.shape}")
dfTrain = dfTrain.drop(columns=dropList, errors='ignore')
dfVal = dfVal.drop(columns=dropList, errors='ignore')
dfTest = dfTest.drop(columns=dropList, errors='ignore')
print(f"after - train:{dfTrain.shape} val:{dfVal.shape} test:{dfTest.shape}")

step 4 : Drop highly correlated pairs (r > 95%)
- why :
    - if two features are same (or highly correlated to each other) they carry redundant information (coliearity) --> keep only one

- fix :
    - correlation
        - -1 < r < 1
        - positive  : one increase --> other increases
        - negative  : one increase --> other decreases
        - zero      : no correlation
    - in our case
        - -0.95 < r < 0.95 --> those two features are higly dependent (|r| < 0.95)
        - correlation with target variable high --> has more valuable information

        $$r = \frac {n \sum xy - \sum x \sum y} {\sqrt {\left[ n \sum x^2 - (\sum x)^2 \right] \left[ n\sum y^2 - (\sum y)^2)\right] }}$$

In [ ]:
import numpy as np
def findHighlyCorrelatedFeatures(dfTrain, verbose=False):
    protected = getProtectedFeatures()
    candidates = [c for c in dfTrain.columns if c not in protected]

    #       x       y       z
    #   x   1       0.3     0
    #   y   0.3     1       0.95
    #   z   0       0.95    1

    corr_matrix = dfTrain[candidates].corr().abs()
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    target_corr = dfTrain[candidates].corrwith(dfTrain['SalePrice']).abs()

    cols_drop = set()

    highCorrPairs = upper_tri.stack()
    highCorrPairs = highCorrPairs[highCorrPairs > 0.95]

    for colA, colB in highCorrPairs.index:
        if colA in cols_drop or colB in cols_drop:
            continue

        if verbose:
            print(f"colinear pair - feature1: {colA} feature2: {colB}")

        if target_corr[colA] < target_corr[colB]:
            cols_drop.add(colA)
        else:
            cols_drop.add(colB)

    dropped_features = list(cols_drop)
    if(verbose):
        print(f"found {len(dropped_features)} higly redundant features: {dropped_features}\n")

    return dropped_features

In [ ]:
itr = 0
while True:
    itr += 1
    print(f"\nIteration: {itr}")
    currentFeatures = dfTrain.shape[1]

    dropList = findHighlyCorrelatedFeatures(dfTrain, True)
    dfTrain, dfVal, dfTest = updateOneHotGrps(oneHotGroup, dropList, dfTrain, dfVal, dfTest, True)

    newFeatures = dfTrain.shape[1]

    if currentFeatures == newFeatures:
        break

In [ ]:
print(f"train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")
dfTrain = dfTrain.drop(columns=dropList, errors='ignore')
dfVal = dfVal.drop(columns=dropList, errors='ignore')
dfTest = dfTest.drop(columns=dropList, errors='ignore')
print(f"train:{dfTrain.shape} validate:{dfVal.shape} test:{dfTest.shape}")

In [ ]:
fh.saveDataSets(dfTrain, dfVal, dfTest, "housingdataset/preprocessing", "07_featureslectionheuristic")